# THOR — Data model and single-`Candidate` pipeline

<div align="center">
<img src="../figures/logo.svg" width="600">
</div>

**Author:** Fabio Ragosta (they/them), fix-term researcher at the University of Naples "Federico II"

**Tutorial 1 of 3.**

THOR (*Fabio Ragosta et al.*) is a Python tool for aggregating and ranking astronomical alerts coming
from multiple brokers (Fink, Lasair, ALeRCE). Before querying a real broker (tutorial 2) and before
combining several brokers with different weights (tutorial 3), this notebook covers:

1. the core data model (`Candidate` and the classes it is built from);
2. how to build a `Candidate` by hand;
3. the internal pipeline that `Hunter.process()` applies to every candidate
   (`FeatureExtractor` → `CrossMatchService` → `CalibrationService` → `RankingService`);
4. which parts of the pipeline are fully implemented and which are still placeholders.


In [1]:
from thor.model import (
    Candidate,
    Coordinates,
    Detection,
    BrokerInfo,
    Classification,
    HostGalaxy,
)
from thor.hunter import Hunter


## 1. The data model

`Candidate` is a `dataclass` that collects everything THOR knows about an object: coordinates,
photometric detections, information about the brokers that reported it, classification, derived
features, and the final ranking.


In [2]:
coords = Coordinates(ra=10.68458, dec=41.26917)
print(coords, "->", coords.tuple())

broker_info = BrokerInfo(broker="Fink", object_id="ZTF21abcxyz")
print(broker_info)

classification = Classification(
    probabilities={"SN Ia": 0.82, "SN II": 0.13, "AGN": 0.05},
    classifier="SuperNNova",
)
print("Most likely class:", classification.best_class, classification.best_probability)


Coordinates(ra=10.68458, dec=41.26917) -> (10.68458, 41.26917)
BrokerInfo(broker='Fink', object_id='ZTF21abcxyz', url=None, retrieved_at=datetime.datetime(2026, 7, 30, 8, 29, 59, 717011), raw=None)
Most likely class: SN Ia 0.82


## 2. Building a `Candidate` by hand

Let's simulate a ZTF alert with a small light curve, an originating broker, and a classification
(as if we had just received it from a broker, before running it through the THOR pipeline).


In [3]:
candidate = Candidate(coordinates=coords)

candidate.add_broker(broker_info)

for mjd, mag, magerr, filt in [
    (59400.1, 19.8, 0.05, "g"),
    (59402.3, 19.1, 0.04, "g"),
    (59404.5, 18.6, 0.03, "r"),
    (59408.7, 18.9, 0.04, "r"),
    (59412.2, 19.5, 0.06, "g"),
]:
    candidate.add_detection(mjd=mjd, mag=mag, magerr=magerr, filt=filt)

candidate.sort_detections()
candidate.classification = classification

print(candidate)
print("Number of detections:", candidate.ndet)
print("First detection:", candidate.first_detection)
print("Last detection:", candidate.last_detection)


Candidate(object='ZTF21abcxyz', class='SN Ia (82.00%)', RA=10.68458, Dec=41.26917, brokers=[Fink])
Number of detections: 5
First detection: Detection(mjd=59400.1, mag=19.8, magerr=0.05, filt='g', snr=None)
Last detection: Detection(mjd=59412.2, mag=19.5, magerr=0.06, filt='g', snr=None)


## 3. The `Hunter.process()` pipeline

`Hunter.process(candidate)` runs four services in sequence:

| Service | What it does today | Status |
|---|---|---|
| `FeatureExtractor` | light-curve statistics (amplitude, duration, cadence, `delta_mag`, ...) | complete |
| `CrossMatchService` | runs a list of registered *matchers* (external catalogs) | empty by default (plugin) |
| `CalibrationService` | copies the best probability into `confidence` | **placeholder** — no real statistical calibration yet (isotonic/Platt/beta planned) |
| `RankingService` | combines weighted partial scores into `ranking.total` | mostly complete — see below |

Let's run it on our candidate.


In [4]:
hunter = Hunter()
processed = hunter.process(candidate)

print("Extracted features:")
for name, value in processed.features.items():
    print(f"  {name}: {value:.3f}")

print()
print("Classification:", processed.classification.best_class,
      f"({processed.classification.best_probability:.0%})")
print("Confidence (post-calibration):", processed.classification.confidence)

print()
r = processed.ranking
print("Ranking:")
print(f"  classification={r.classification:.2f}  lightcurve={r.lightcurve:.2f}  "
      f"temporal={r.temporal:.2f}  host={r.host:.2f}  agreement={r.agreement:.2f}  "
      f"rarity={r.rarity:.2f}")
print(f"  TOTAL = {r.total:.4f}")


Extracted features:
  n_detections: 5.000
  min_mag: 18.600
  max_mag: 19.800
  amplitude: 1.200
  mean_mag: 19.180
  std_mag: 0.426
  duration: 12.100
  cadence: 3.025
  delta_mag: -0.300

Classification: SN Ia (82%)
Confidence (post-calibration): 0.82

Ranking:
  classification=0.82  lightcurve=0.25  temporal=0.60  host=0.00  agreement=0.50  rarity=0.40
  TOTAL = 0.5945


Note on `agreement`: with a single broker (as here) there is nothing to compare, so
`agreement_score()` returns a neutral `0.5` by design. We'll see it produce a real, varying value
once several brokers are combined in tutorial 3.


## 4. Extending cross-match

`CrossMatchService` does nothing by default: it is a registry of functions (*matchers*) that the user
plugs in. Each matcher receives a `Candidate` and returns an updated one; if it raises an exception,
THOR records it in `candidate.metadata["crossmatch_errors"]` and keeps running the remaining matchers,
without interrupting the pipeline.


In [5]:
def dummy_host_matcher(candidate):
    """Example cross-match: attaches a dummy host galaxy (e.g. from NED/SDSS)."""
    candidate.host = HostGalaxy(
        name="NGC 000X",
        redshift=0.015,
        separation_arcsec=3.2,
        catalog="NED",
    )
    return candidate

hunter.crossmatch.register(dummy_host_matcher)

processed = hunter.process(candidate)
print("Host:", processed.host)
print("Ranking host score:", processed.ranking.host)
print("Updated total ranking:", processed.ranking.total)


Host: HostGalaxy(name='NGC 000X', redshift=0.015, separation_arcsec=3.2, catalog='NED')
Ranking host score: 1.0
Updated total ranking: 0.6445


## 5. What this pipeline still doesn't do

- **Cross-match**: no catalog is wired in by default — it must be registered explicitly (as above).
- **Calibration**: `CalibrationService` only copies the best probability into `confidence`; it does not
  yet apply any real statistical calibration.
- **`host_score` and `rarity_score`**: simple rule-based heuristics, not yet backed by statistical models
  or reference catalogs.

One component that used to be a placeholder and no longer is: **`agreement_score`** now computes the mean
pairwise cosine similarity between the probability distributions reported by every broker that classified
the candidate (falling back to a neutral `0.5` with 0 or 1 broker, as seen above). With a single broker,
as in this notebook, you'll always see the neutral value — tutorial 3 shows it producing real, varying
scores once multiple brokers are combined.

## Next notebook

**Tutorial 2** shows how to query a real broker with `Hunter.load()` / `Hunter.search()` /
`Hunter.cone_search()` / `Hunter.get()`, including a known bug in `cone_search()`.
